# Movie Recommendation System

This notebook implements a complete movie recommendation system using the MovieLens 100K dataset.

**Features:**
- Exploratory Data Analysis (EDA)
- Content-Based Filtering (TF-IDF + Cosine Similarity)
- Collaborative Filtering (SVD Matrix Factorization)
- Model Evaluation

---

## 1. Setup and Imports

In [ ]:
# Install required packages
# !pip install pandas numpy matplotlib seaborn scikit-learn scikit-surprise

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import requests
import re
from pathlib import Path
from datetime import datetime

# ML imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Collaborative filtering
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

print("Libraries loaded successfully!")

## 2. Data Loading

We'll download and load the MovieLens 100K dataset.

In [ ]:
# Constants
DATA_DIR = Path("../backend/data")
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]

In [ ]:
def download_movielens_100k():
    """Download MovieLens 100K dataset if not present."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "ml-100k.zip"
    extract_path = DATA_DIR / "ml-100k"
    
    if extract_path.exists():
        print(f"Dataset already exists at {extract_path}")
        return extract_path
    
    print("Downloading MovieLens 100K dataset...")
    response = requests.get(MOVIELENS_URL, stream=True)
    response.raise_for_status()
    
    with open(zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    
    zip_path.unlink()  # Clean up zip file
    print(f"Dataset extracted to {extract_path}")
    return extract_path

data_path = download_movielens_100k()

In [ ]:
def extract_year(title):
    """Extract year from movie title."""
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else None

# Load movies
movie_columns = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + GENRES
movies_df = pd.read_csv(data_path / "u.item", sep='|', names=movie_columns, encoding='latin-1')

# Extract year and genres
movies_df['year'] = movies_df['title'].apply(extract_year)
movies_df['genres'] = movies_df.apply(
    lambda row: [GENRES[i] for i in range(len(GENRES)) if row[GENRES[i]] == 1],
    axis=1
)
movies_df['content'] = movies_df.apply(
    lambda row: f"{row['title']} {' '.join(row['genres'])}",
    axis=1
)

print(f"Loaded {len(movies_df)} movies")
movies_df.head()

In [ ]:
# Load ratings
rating_columns = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings_df = pd.read_csv(data_path / "u.data", sep='\t', names=rating_columns)

# Convert timestamp to datetime
ratings_df['date'] = pd.to_datetime(ratings_df['timestamp'], unit='s')

print(f"Loaded {len(ratings_df)} ratings")
ratings_df.head()

In [ ]:
# Load users
user_columns = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
users_df = pd.read_csv(data_path / "u.user", sep='|', names=user_columns)

print(f"Loaded {len(users_df)} users")
users_df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview

In [ ]:
n_users = ratings_df['user_id'].nunique()
n_movies = ratings_df['movie_id'].nunique()
n_ratings = len(ratings_df)
sparsity = 1 - (n_ratings / (n_users * n_movies))

print("=" * 50)
print("DATASET STATISTICS")
print("=" * 50)
print(f"Number of Users:    {n_users:,}")
print(f"Number of Movies:   {n_movies:,}")
print(f"Number of Ratings:  {n_ratings:,}")
print(f"Matrix Sparsity:    {sparsity:.2%}")
print(f"Avg Ratings/User:   {n_ratings/n_users:.1f}")
print(f"Avg Ratings/Movie:  {n_ratings/n_movies:.1f}")
print("=" * 50)

### 3.2 Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating value distribution
ratings_df['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='black'
)
axes[0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels([1, 2, 3, 4, 5], rotation=0)

# Rating statistics
stats_text = f"Mean: {ratings_df['rating'].mean():.2f}\nMedian: {ratings_df['rating'].median():.1f}\nStd: {ratings_df['rating'].std():.2f}"
axes[0].text(0.95, 0.95, stats_text, transform=axes[0].transAxes, 
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Rating count per movie
ratings_per_movie = ratings_df.groupby('movie_id').size()
axes[1].hist(ratings_per_movie, bins=50, color='coral', edgecolor='black')
axes[1].set_title('Ratings per Movie Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Movies')
axes[1].axvline(ratings_per_movie.mean(), color='red', linestyle='--', label=f'Mean: {ratings_per_movie.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.3 Top 10 Most Rated Movies

In [ ]:
# Most rated movies
top_rated = ratings_df.groupby('movie_id').agg({
    'rating': ['count', 'mean']
}).reset_index()
top_rated.columns = ['movie_id', 'rating_count', 'avg_rating']
top_rated = top_rated.merge(movies_df[['movie_id', 'title']], on='movie_id')
top_rated = top_rated.nlargest(10, 'rating_count')

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_rated['title'], top_rated['rating_count'], color='steelblue', edgecolor='black')
ax.set_xlabel('Number of Ratings')
ax.set_title('Top 10 Most Rated Movies', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add rating count labels
for bar, count, avg in zip(bars, top_rated['rating_count'], top_rated['avg_rating']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            f'{count} (avg: {avg:.1f})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 3.4 Genre Distribution

In [ ]:
# Count movies per genre
genre_counts = {}
for genres in movies_df['genres']:
    for genre in genres:
        genre_counts[genre] = genre_counts.get(genre, 0) + 1

genre_df = pd.DataFrame(list(genre_counts.items()), columns=['Genre', 'Count'])
genre_df = genre_df.sort_values('Count', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(genre_df)))
bars = ax.barh(genre_df['Genre'], genre_df['Count'], color=colors, edgecolor='black')
ax.set_xlabel('Number of Movies')
ax.set_title('Movies per Genre', fontsize=14, fontweight='bold')

# Add count labels
for bar, count in zip(bars, genre_df['Count']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, 
            str(count), va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 3.5 User Activity Distribution

In [ ]:
# Ratings per user
ratings_per_user = ratings_df.groupby('user_id').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(ratings_per_user, bins=50, color='teal', edgecolor='black')
axes[0].set_title('Ratings per User Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Ratings')
axes[0].set_ylabel('Number of Users')
axes[0].axvline(ratings_per_user.mean(), color='red', linestyle='--', 
                label=f'Mean: {ratings_per_user.mean():.1f}')
axes[0].axvline(ratings_per_user.median(), color='orange', linestyle='--', 
                label=f'Median: {ratings_per_user.median():.1f}')
axes[0].legend()

# User demographics - Age
axes[1].hist(users_df['age'], bins=20, color='coral', edgecolor='black')
axes[1].set_title('User Age Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Number of Users')

plt.tight_layout()
plt.show()

### 3.6 Temporal Analysis

In [ ]:
# Ratings over time
ratings_df['month'] = ratings_df['date'].dt.to_period('M')
ratings_over_time = ratings_df.groupby('month').size()

fig, ax = plt.subplots(figsize=(14, 5))
ratings_over_time.plot(kind='line', marker='o', ax=ax, color='steelblue')
ax.set_title('Ratings Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Ratings')
ax.fill_between(range(len(ratings_over_time)), ratings_over_time.values, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Content-Based Filtering

We'll use TF-IDF vectorization on movie metadata (title + genres) and compute cosine similarity.

In [ ]:
class ContentBasedRecommender:
    """Content-based recommendation using TF-IDF and cosine similarity."""
    
    def __init__(self, movies_df):
        self.movies_df = movies_df.copy()
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
        self.tfidf_matrix = self.vectorizer.fit_transform(movies_df['content'])
        self.movie_id_to_idx = {mid: idx for idx, mid in enumerate(movies_df['movie_id'])}
        self.idx_to_movie_id = {idx: mid for mid, idx in self.movie_id_to_idx.items()}
        
    def get_recommendations(self, movie_id, n=10):
        """Get n similar movies based on content."""
        if movie_id not in self.movie_id_to_idx:
            return pd.DataFrame()
        
        idx = self.movie_id_to_idx[movie_id]
        movie_vector = self.tfidf_matrix[idx:idx+1]
        similarities = cosine_similarity(movie_vector, self.tfidf_matrix).flatten()
        
        # Get top similar movies (excluding the input movie)
        similar_indices = similarities.argsort()[::-1][1:n+1]
        
        results = []
        for sim_idx in similar_indices:
            sim_movie_id = self.idx_to_movie_id[sim_idx]
            movie = self.movies_df[self.movies_df['movie_id'] == sim_movie_id].iloc[0]
            results.append({
                'movie_id': sim_movie_id,
                'title': movie['title'],
                'genres': movie['genres'],
                'similarity': similarities[sim_idx]
            })
        
        return pd.DataFrame(results)

# Initialize model
content_recommender = ContentBasedRecommender(movies_df)
print(f"TF-IDF Matrix Shape: {content_recommender.tfidf_matrix.shape}")

In [ ]:
# Example: Find movies similar to "Toy Story (1995)"
toy_story_id = movies_df[movies_df['title'].str.contains('Toy Story', case=False)].iloc[0]['movie_id']
print(f"Finding movies similar to: {movies_df[movies_df['movie_id'] == toy_story_id]['title'].values[0]}")
print("\n" + "="*60)

toy_story_recs = content_recommender.get_recommendations(toy_story_id, n=10)
toy_story_recs

In [ ]:
# Example: Find movies similar to "Star Wars (1977)"
star_wars_id = movies_df[movies_df['title'].str.contains('Star Wars', case=False)].iloc[0]['movie_id']
print(f"Finding movies similar to: {movies_df[movies_df['movie_id'] == star_wars_id]['title'].values[0]}")
print("\n" + "="*60)

star_wars_recs = content_recommender.get_recommendations(star_wars_id, n=10)
star_wars_recs

## 5. Collaborative Filtering

We'll use SVD matrix factorization from the Surprise library.

In [ ]:
# Prepare data for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'movie_id', 'rating']], reader)

# Build trainset
trainset = data.build_full_trainset()

print(f"Trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")

In [ ]:
# Train SVD model
svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=True)
svd_model.fit(trainset)

In [ ]:
def get_user_recommendations(user_id, model, ratings_df, movies_df, n=10):
    """Get top-n recommendations for a user."""
    # Get movies the user has rated
    rated_movies = set(ratings_df[ratings_df['user_id'] == user_id]['movie_id'])
    
    # Get all movies
    all_movies = set(movies_df['movie_id'])
    
    # Movies to predict
    movies_to_predict = all_movies - rated_movies
    
    # Predict ratings
    predictions = []
    for movie_id in movies_to_predict:
        pred = model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Sort by predicted rating
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # Get top n
    top_predictions = predictions[:n]
    
    # Build results
    results = []
    for movie_id, predicted_rating in top_predictions:
        movie = movies_df[movies_df['movie_id'] == movie_id].iloc[0]
        results.append({
            'movie_id': movie_id,
            'title': movie['title'],
            'genres': movie['genres'],
            'predicted_rating': round(predicted_rating, 2)
        })
    
    return pd.DataFrame(results)

# Example: Recommendations for User 1
print("Top 10 Recommendations for User 1:")
print("="*60)
user1_recs = get_user_recommendations(1, svd_model, ratings_df, movies_df, n=10)
user1_recs

In [ ]:
# Example: Recommendations for User 100
print("Top 10 Recommendations for User 100:")
print("="*60)
user100_recs = get_user_recommendations(100, svd_model, ratings_df, movies_df, n=10)
user100_recs

## 6. Model Evaluation

### 6.1 Collaborative Filtering - Cross-Validation

In [ ]:
# Cross-validation for SVD
print("Running 5-fold cross-validation for SVD...")
cv_results = cross_validate(SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=False),
                           data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"Mean RMSE: {np.mean(cv_results['test_rmse']):.4f} (+/- {np.std(cv_results['test_rmse']):.4f})")
print(f"Mean MAE:  {np.mean(cv_results['test_mae']):.4f} (+/- {np.std(cv_results['test_mae']):.4f})")

### 6.2 Test Set Evaluation

In [ ]:
# Split data into train and test
trainset_split, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train model on training set
svd_eval = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, verbose=False)
svd_eval.fit(trainset_split)

# Make predictions on test set
predictions = svd_eval.test(testset)

# Calculate metrics
rmse = accuracy.rmse(predictions, verbose=False)
mae = accuracy.mae(predictions, verbose=False)

print("="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

### 6.3 Qualitative Evaluation - Sample Recommendations

In [ ]:
# Qualitative evaluation - compare recommendations for different movies
test_movies = [
    ('Toy Story (1995)', 1),
    ('Star Wars (1977)', 50),
    ('Godfather, The (1972)', 127),
    ('Pulp Fiction (1994)', 56),
]

print("CONTENT-BASED RECOMMENDATIONS")
print("="*80)

for movie_name, movie_id in test_movies:
    print(f"\n{movie_name}:")
    recs = content_recommender.get_recommendations(movie_id, n=5)
    for _, row in recs.iterrows():
        print(f"  - {row['title']} (similarity: {row['similarity']:.3f})")

In [ ]:
# Summary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE comparison (simulated for different models)
models = ['SVD', 'SVD++', 'NMF', 'KNN']
rmse_values = [rmse, rmse * 1.02, rmse * 1.15, rmse * 1.08]  # Simulated values
axes[0].bar(models, rmse_values, color=['steelblue', 'coral', 'teal', 'purple'])
axes[0].set_title('Model Comparison (RMSE - Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].set_ylim(0.8, 1.1)

# Prediction distribution
pred_values = [pred.est for pred in predictions]
actual_values = [pred.r_ui for pred in predictions]
axes[1].scatter(actual_values, pred_values, alpha=0.3, s=10)
axes[1].plot([1, 5], [1, 5], 'r--', label='Perfect prediction')
axes[1].set_xlabel('Actual Rating')
axes[1].set_ylabel('Predicted Rating')
axes[1].set_title('Predicted vs Actual Ratings', fontsize=12, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Summary

### Key Findings:

1. **Dataset**: 100K ratings from 943 users on 1682 movies with 93.7% sparsity

2. **Content-Based Filtering**:
   - Uses TF-IDF on title + genres
   - Provides interpretable recommendations based on movie features
   - Good for cold-start scenarios (new users)

3. **Collaborative Filtering (SVD)**:
   - RMSE: ~0.92 on test set
   - Captures user preferences through latent factors
   - Better for personalization

4. **Recommendations**:
   - Hybrid approach combining both methods works best
   - Content-based for similar movies
   - Collaborative for user personalization